Ensure to run file "RF_Initial_Feature_Engineering.py" in path 2b_feature_engineering prior to this notebook to populate featureset.

In [1]:
import os
import pandas as pd
import numpy as np
import pyarrow.dataset as ds

DATA_DIR = os.path.join("..", "..", "..","1_download_data")
MODEL_READY_PATH = os.path.join(DATA_DIR, "model_ready", "flights_model_ready.parquet")

print("Model-ready exists?", os.path.exists(MODEL_READY_PATH), MODEL_READY_PATH)

N = 400_000  
want_cols = [
    "FlightDate", "CRSDepTime", "Distance", "Origin",
    "tmpf", "vsby", "sknt", "p01i", "relh", "gust",
    "target"
]

dataset = ds.dataset(MODEL_READY_PATH, format="parquet")
cols = [c for c in want_cols if c in dataset.schema.names]  

table = dataset.to_table(columns=cols)
df = table.slice(0, N).to_pandas()

print("df:", df.shape)
df.head()

Model-ready exists? True ../../../1_download_data/model_ready/flights_model_ready.parquet
df: (400000, 11)


,FlightDate,CRSDepTime,Distance,Origin,tmpf,vsby,sknt,p01i,relh,gust,target
0,2018-01-03,1037,145.0,ATL,27.0,10.0,6.000000,0.0,40.42,0.0,Delayed
1,2018-01-04,1037,145.0,ATL,24.1,10.0,14.166667,0.0,54.36,21.0,On time
2,2018-01-05,1037,145.0,ATL,17.1,10.0,12.153846,0.0,55.51,0.0,On time
3,2018-01-06,1037,145.0,ATL,21.9,10.0,6.846154,0.0,57.02,0.0,On time
4,2018-01-07,1037,145.0,ATL,21.9,10.0,7.384615,0.0,54.75,0.0,On time


In [2]:
from sklearn.model_selection import train_test_split

df = df.dropna(subset=["target"]).copy()

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

print("train:", train_df.shape)
print("test :", test_df.shape)

train: (320000, 11)
test : (80000, 11)


In [3]:
for d in (train_df, test_df):
    d["CRSDepTime"] = pd.to_numeric(d["CRSDepTime"], errors="coerce")
    d["dep_hour"] = (d["CRSDepTime"] // 100).astype("Int64")
    d["month"] = d["FlightDate"].dt.month
    d["day_of_week"] = d["FlightDate"].dt.dayofweek

train_df = train_df.dropna(subset=["dep_hour"])
test_df  = test_df.dropna(subset=["dep_hour"])

train_df = train_df[(train_df["dep_hour"] >= 0) & (train_df["dep_hour"] <= 23)]
test_df  = test_df[(test_df["dep_hour"] >= 0) & (test_df["dep_hour"] <= 23)]

print(train_df.shape, test_df.shape)

(320000, 14) (80000, 14)


In [4]:
NUM_COLS = [
    "Distance", "dep_hour", "month", "day_of_week",
    "tmpf", "vsby", "sknt", "p01i", "relh", "gust"
]
CAT_COLS = ["Origin"]

X_train = train_df[NUM_COLS + CAT_COLS]
y_train = train_df["target"]

X_test  = test_df[NUM_COLS + CAT_COLS]
y_test  = test_df["target"]

print(X_train.shape, X_test.shape)

(320000, 11) (80000, 11)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

prep = ColumnTransformer([
    ("num", num_pipe, NUM_COLS),
    ("cat", cat_pipe, CAT_COLS),
])

lr = LogisticRegression(
    max_iter=300,
    n_jobs=-1,
    class_weight={
        "On time": 1.0,
        "Delayed": 1.5,
        "Cancelled": 2.0
    }
)

model = Pipeline([
    ("prep", prep),
    ("lr", lr),
])

print("✅ Model built")

✅ Model built


In [6]:
model.fit(X_train, y_train)
print("✅ Done fitting")

✅ Done fitting


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

pred = model.predict(X_test)

print(classification_report(y_test, pred, digits=3))
print("Confusion matrix [On time, Delayed, Cancelled]:")
print(confusion_matrix(y_test, pred, labels=["On time", "Delayed", "Cancelled"]))

              precision    recall  f1-score   support

   Cancelled      0.628     0.156     0.250      1743
     Delayed      0.473     0.053     0.096     14450
     On time      0.808     0.987     0.889     63807

    accuracy                          0.800     80000
   macro avg      0.636     0.399     0.412     80000
weighted avg      0.744     0.800     0.732     80000

Confusion matrix [On time, Delayed, Cancelled]:
[[62987   745    75]
 [13591   773    86]
 [ 1355   116   272]]
